# LAB-D4-04: Capstone - The Neural Network Investigation

**Purpose:** Investigate one anonymous routing-code model profile, commit a falsifiable diagnosis, spend one primary experiment, unlock test evidence only after the decision, and defend the complete evidence chain.

**Objectives:** `OBJ-D4-01` through `OBJ-D4-08`, with primary evidence for `OBJ-D4-08`  
**Estimated duration:** 95 minutes work plus defense; complete single-profile CPU path under 12 minutes  
**Prerequisites:** all prior core labs, especially `LAB-D4-01`, `LAB-D4-02`, `LAB-D4-03`, and [LESSON-D4-08](../day-4/student-guide/day-4-student-guide.md#lesson-d4-08---capstone-defend-the-evidence-chain)  
**Environment:** CPU; PyTorch, NumPy, matplotlib, scikit-learn; fixed local digits data and cached baseline/recovery evidence; no network

Read the [Capstone Student Guide](capstone-student-guide.md) and [Capstone Rubric](capstone-rubric.md). This is the sole participant implementation of `LAB-D4-04`.

In [ ]:
import hashlib
import json
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sklearn
import torch
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

torch.set_num_threads(1)
started_capstone = time.perf_counter()

def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "courseware/shared/data/day-4/manifest.json").exists(): return candidate
    raise FileNotFoundError("Run from the repository checkout with courseware/shared/data/day-4 present.")

ROOT = find_repo_root(); DATA_DIR = ROOT / "courseware/shared/data/day-4"
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print(f"Python {platform.python_version()} | torch {torch.__version__} | scikit-learn {sklearn.__version__}")
print("Required device: CPU. Fixed seeds aid repeatability but do not imply cross-device bitwise equality.")

## Case Brief: Routing-Code Recognition

The input is an 8x8 handwritten digit image. A routing system predicts one of ten codes. Misrouting a weak class is costly, but overall quality, generalization, latency, and size still matter. Profiles `A`, `B`, and `C` are anonymous; names, filenames, colors, and metadata do not state a cause.

In [ ]:
PROFILE_ID = "A"  # TODO: enter the assigned profile: A, B, or C.
assert PROFILE_ID in {"A", "B", "C"}
selected_baseline_record = manifest["artifacts"][f"profile_{PROFILE_ID}_baseline"]
selected_baseline_path = DATA_DIR / selected_baseline_record["path"]
assert hashlib.sha256(selected_baseline_path.read_bytes()).hexdigest() == selected_baseline_record["sha256"]
baseline = np.load(selected_baseline_path, allow_pickle=False)
case_target = str(baseline["case_target"])
print("Assigned profile:", PROFILE_ID)
print("Assigned target:", case_target)

## Fixed Split and Test Lock

The split hash is evidence. Training and validation arrays are bound now. Test indices remain behind a function gate until the primary experiment is interpreted and the decision record is complete.

In [ ]:
digits = load_digits(); X_all = digits.images.astype(np.float32); y_all = digits.target.astype(np.int64)
splits = np.load(DATA_DIR / manifest["artifacts"]["fixed_splits"]["path"], allow_pickle=False)
train_indices, val_indices = splits["train_indices"], splits["val_indices"]
_locked_test_indices = splits["test_indices"]
split_hash = str(splits["split_hash"])
assert split_hash == manifest["fixed_split"]["hash"]
X_all_tensor = torch.tensor(X_all / 16.0, dtype=torch.float32)
y_all_tensor = torch.tensor(y_all, dtype=torch.long)
TEST_AUTHORIZED = False

def access_test_split():
    if not TEST_AUTHORIZED: raise PermissionError("Test is locked until the decision gate passes.")
    return X_all_tensor[_locked_test_indices], y_all_tensor[_locked_test_indices]

try:
    access_test_split(); raise AssertionError("Test lock failed")
except PermissionError as error:
    print("Expected lock diagnostic:", error)
print("Fixed split hash:", split_hash)

In [ ]:
import random
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cpu")

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

class DigitsMLP(nn.Module):
    def __init__(self, hidden=(32,), dropout=0.0):
        super().__init__(); layers = [nn.Flatten()]; input_dim = 64
        for width in hidden:
            layers.extend([nn.Linear(input_dim, width), nn.ReLU()])
            if dropout > 0: layers.append(nn.Dropout(dropout))
            input_dim = width
        layers.append(nn.Linear(input_dim, 10)); self.network = nn.Sequential(*layers)
    def forward(self, inputs): return self.network(inputs)

def evaluate_model(model, X, y):
    model.eval()
    with torch.inference_mode():
        logits = model(X); probabilities = torch.softmax(logits, dim=1).cpu().numpy()
    predictions = probabilities.argmax(1); targets = y.cpu().numpy()
    return {
        "loss": float(nn.CrossEntropyLoss()(logits, y).item()),
        "accuracy": float(accuracy_score(targets, predictions)),
        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),
        "probabilities": probabilities, "predictions": predictions,
    }

def run_experiment(config, train_indices, val_indices):
    set_all_seeds(config["seed"])
    model = DigitsMLP(tuple(config["hidden"]), config["dropout"]).to(DEVICE)
    optimizer_class = {"adam": torch.optim.Adam, "adamw": torch.optim.AdamW, "sgd": torch.optim.SGD}[config["optimizer_name"]]
    extra = {"momentum": 0.9} if config["optimizer_name"] == "sgd" else {}
    optimizer = optimizer_class(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"], **extra)
    selected_labels = y_all[train_indices]
    if config["class_weight"]:
        counts = np.bincount(selected_labels, minlength=10); weights = len(selected_labels) / (10 * np.maximum(counts, 1))
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32))
    else: loss_fn = nn.CrossEntropyLoss()
    generator = torch.Generator().manual_seed(4403 + config["seed"])
    loader = DataLoader(TensorDataset(X_all_tensor[train_indices], y_all_tensor[train_indices]), batch_size=config["batch_size"], shuffle=True, generator=generator, num_workers=0)
    history = {name: [] for name in ["train_loss", "val_loss", "train_accuracy", "val_accuracy"]}; started = time.perf_counter()
    for _ in range(config["epochs"]):
        model.train()
        for X_batch, y_batch in loader:
            optimizer.zero_grad(); loss = loss_fn(model(X_batch), y_batch); loss.backward(); optimizer.step()
        train_metrics = evaluate_model(model, X_all_tensor[train_indices], y_all_tensor[train_indices])
        val_metrics = evaluate_model(model, X_all_tensor[val_indices], y_all_tensor[val_indices])
        for prefix, values in [("train", train_metrics), ("val", val_metrics)]:
            history[f"{prefix}_loss"].append(values["loss"]); history[f"{prefix}_accuracy"].append(values["accuracy"])
    return {"model": model, "config": dict(config), "history": history,
            "train": evaluate_model(model, X_all_tensor[train_indices], y_all_tensor[train_indices]),
            "validation": evaluate_model(model, X_all_tensor[val_indices], y_all_tensor[val_indices]),
            "training_seconds": time.perf_counter() - started,
            "parameter_count": sum(p.numel() for p in model.parameters()),
            "parameter_bytes": sum(p.numel() * p.element_size() for p in model.parameters())}

def benchmark_batch1(model, sample, warmup=20, repeats=100):
    model.eval(); timings = []
    with torch.inference_mode():
        for _ in range(warmup): model(sample)
        for _ in range(repeats):
            started = time.perf_counter(); model(sample); timings.append((time.perf_counter() - started) * 1000)
    return {"median_ms": float(np.median(timings)), "p90_ms": float(np.percentile(timings, 90)), "warmup": warmup, "repeats": repeats}

In [ ]:
case_train_indices = np.asarray(baseline["case_train_indices"], dtype=np.int64)
BASELINE_CONFIG = json.loads(str(baseline["baseline_config_json"]))
expected_config_fields = {
    "hidden", "dropout", "optimizer_name", "learning_rate", "weight_decay",
    "class_weight", "epochs", "batch_size", "seed",
}
assert set(BASELINE_CONFIG) == expected_config_fields
assert len(case_train_indices) == len(np.unique(case_train_indices))
assert np.all(np.isin(case_train_indices, train_indices))
print("Selected case contract loaded; training examples:", len(case_train_indices))

## Baseline Evidence Board: Facts First

Inspect curves, metrics, confusion, high-confidence errors, runtime, and size before naming a cause. Facts and inferences belong in different fields.

In [ ]:
baseline_confusion = baseline["confusion"]
baseline_summary = {
    "train_accuracy": float(baseline["train_accuracy"][-1]),
    "validation_accuracy": float(baseline["val_accuracy"][-1]),
    "validation_macro_f1": float(f1_score(baseline["val_labels"], baseline["val_predictions"], average="macro", zero_division=0)),
    "worst_class_recall": float(baseline["class_recall"].min()),
    "training_seconds": float(baseline["training_seconds"]),
    "parameter_bytes": int(baseline["parameter_bytes"]),
    "batch1_latency_ms": float(baseline["latency_ms_median"]),
}
print("Baseline facts:", baseline_summary)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(baseline["epochs"], baseline["train_loss"], label="train"); axes[0].plot(baseline["epochs"], baseline["val_loss"], label="validation"); axes[0].set(title="Loss", xlabel="epoch"); axes[0].legend()
axes[1].plot(baseline["epochs"], baseline["train_accuracy"], label="train"); axes[1].plot(baseline["epochs"], baseline["val_accuracy"], label="validation"); axes[1].set(title="Accuracy", xlabel="epoch", ylim=(0,1.02)); axes[1].legend()
axes[2].imshow(baseline_confusion, cmap="Greys"); axes[2].set(title="Validation confusion", xlabel="predicted", ylabel="actual", xticks=range(10), yticks=range(10))
plt.tight_layout(); plt.show()

In [ ]:
val_confidence = baseline["val_probabilities"].max(axis=1)
error_positions = np.flatnonzero(baseline["val_predictions"] != baseline["val_labels"])
shown = error_positions[np.argsort(val_confidence[error_positions])[::-1]][:8]
fig, axes = plt.subplots(2, 4, figsize=(9, 5))
for position, ax in zip(shown, axes.ravel()):
    sample_id = int(baseline["val_sample_ids"][position]); ax.imshow(X_all[sample_id], cmap="gray_r")
    ax.set_title(f"id {sample_id} | y={baseline['val_labels'][position]} p={baseline['val_predictions'][position]}\nscore={val_confidence[position]:.2f}", fontsize=8); ax.axis("off")
plt.suptitle("Selected validation errors; inspect without treating them as prevalence")
plt.tight_layout(); plt.show()

## Request Exactly One Evidence Card

Choose one: `class_counts`, `curve_dynamics`, `confidence_errors`, or `resource_profile`. Explain why it separates your top two hypotheses before revealing it.

In [ ]:
evidence_request = {"card": "", "top_two_hypotheses": "", "why_discriminating": ""}
assert evidence_request["card"] in {"class_counts", "curve_dynamics", "confidence_errors", "resource_profile"}
assert all(value.strip() for value in evidence_request.values())

EVIDENCE_CARDS = {
    "class_counts": baseline["train_class_counts"].tolist(),
    "curve_dynamics": {"last_train_loss": float(baseline["train_loss"][-1]), "last_val_loss": float(baseline["val_loss"][-1]), "val_loss_range": float(np.ptp(baseline["val_loss"]))},
    "confidence_errors": {"errors": int(len(error_positions)), "errors_above_0_60": int(np.sum(val_confidence[error_positions] >= 0.60))},
    "resource_profile": {key: baseline_summary[key] for key in ["training_seconds", "parameter_bytes", "batch1_latency_ms"]},
}
print("Requested card:", evidence_request["card"], EVIDENCE_CARDS[evidence_request["card"]])

## Diagnosis Gate

Rank data/composition, generalization/capacity, optimization, architecture, evaluation, and compute explanations. Name evidence that supports the leader and evidence that would make it wrong. The gate scores reasoning, not a secret label.

In [ ]:
diagnosis_gate = {
    "ranked_hypotheses": "", "primary_limitation": "", "supporting_metric": "",
    "supporting_curve_or_slice": "", "competing_explanation": "", "disconfirming_evidence": "",
    "target_connection": "", "confidence_and_limit": "",
}
assert all(value.strip() for value in diagnosis_gate.values()), "Complete the diagnosis and disconfirming-evidence gate before spending the run."

## Choose One Primary Intervention

Allowed major fields: `hidden`, `dropout`, `optimizer_name`, `learning_rate`, `weight_decay`, or `class_weight`. Change exactly one. Predict observations in metrics, curves/slices, and resources before running.

In [ ]:
MAJOR_FIELDS = {"hidden", "dropout", "optimizer_name", "learning_rate", "weight_decay", "class_weight"}
PRIMARY_CONFIG = dict(BASELINE_CONFIG)
# TODO: change exactly one allowed major field.

changed_fields = [field for field in MAJOR_FIELDS if PRIMARY_CONFIG[field] != BASELINE_CONFIG[field]]
assert len(changed_fields) == 1, f"One primary change required; received {changed_fields}"
assert PRIMARY_CONFIG["epochs"] <= 70
intervention_prediction = {
    "changed_field": changed_fields[0], "mechanism": "", "predicted_metric": "",
    "predicted_curve_or_slice": "", "predicted_resource_effect": "", "rejection_rule": "",
}
assert all(str(value).strip() for value in intervention_prediction.values())

## Run the Primary Experiment

The default path trains from a fresh model/optimizer/loader. Set recovery only after a technical failure and only after the diagnosis/intervention gates. Recovery succeeds only when the selected config fingerprint matches the anonymous cached run; it never supplies test evidence.

In [ ]:
def config_fingerprint(config): return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()

USE_RECOVERY_AFTER_TECHNICAL_FAILURE = False
primary_run = None; recovery_used = False
if not USE_RECOVERY_AFTER_TECHNICAL_FAILURE:
    primary_run = run_experiment(PRIMARY_CONFIG, case_train_indices, val_indices)
    primary_validation = primary_run["validation"]
    primary_history = primary_run["history"]
    primary_resources = {"training_seconds": primary_run["training_seconds"], "parameter_bytes": primary_run["parameter_bytes"], **benchmark_batch1(primary_run["model"], X_all_tensor[val_indices[:1]])}
else:
    recovery_record = manifest["artifacts"][f"profile_{PROFILE_ID}_recovery"]
    assert config_fingerprint(PRIMARY_CONFIG) == recovery_record["selection_fingerprint"], "No cached recovery matches this already-committed primary config."
    recovery = np.load(DATA_DIR / recovery_record["path"], allow_pickle=False); recovery_used = True
    primary_validation = {"accuracy": float(recovery["val_accuracy"][-1]), "macro_f1": float(f1_score(recovery["val_labels"], recovery["val_predictions"], average="macro", zero_division=0)), "predictions": recovery["val_predictions"], "probabilities": recovery["val_probabilities"]}
    primary_history = {key: recovery[key] for key in ["train_loss", "val_loss", "train_accuracy", "val_accuracy"]}
    primary_resources = {"training_seconds": float(recovery["training_seconds"]), "parameter_bytes": int(recovery["parameter_bytes"]), "median_ms": float(recovery["latency_ms_median"]), "p90_ms": float(recovery["latency_ms_p90"]), "warmup": int(recovery["latency_warmup"]), "repeats": int(recovery["latency_repeats"])}
print("Primary validation:", {key: primary_validation[key] for key in ["accuracy", "macro_f1"]}); print("Recovery used:", recovery_used)

In [ ]:
primary_confusion = confusion_matrix(y_all[val_indices], primary_validation["predictions"], labels=np.arange(10))
primary_recall = np.diag(primary_confusion) / np.maximum(primary_confusion.sum(1), 1)
comparison = {
    "accuracy_delta": primary_validation["accuracy"] - baseline_summary["validation_accuracy"],
    "macro_f1_delta": primary_validation["macro_f1"] - baseline_summary["validation_macro_f1"],
    "worst_recall_delta": float(primary_recall.min()) - baseline_summary["worst_class_recall"],
    "gap_before": baseline_summary["train_accuracy"] - baseline_summary["validation_accuracy"],
    "gap_after": float(primary_history["train_accuracy"][-1]) - primary_validation["accuracy"],
}
print("Comparison:", comparison)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(baseline["epochs"], baseline["val_accuracy"], label="baseline"); axes[0].plot(range(1,len(primary_history["val_accuracy"])+1), primary_history["val_accuracy"], label="primary")
axes[0].set(title="Aligned validation accuracy", xlabel="epoch", ylabel="accuracy", ylim=(0,1.02)); axes[0].legend()
axes[1].bar(["baseline", "primary"], [baseline_summary["worst_class_recall"], float(primary_recall.min())], color=["#3b6c8e", "#9b5d42"])
axes[1].set(title="Worst-class recall", ylim=(0,1.02)); plt.tight_layout(); plt.show()

## Serialize the Ledger and Evidence Board

Separate facts, inference, prediction, observed result, and interpretation. No test evidence belongs here yet.

In [ ]:
experiment_ledger = {
    "schema": 1, "profile": PROFILE_ID, "split_hash": split_hash, "data_source": "sklearn.datasets.load_digits",
    "device": str(DEVICE), "versions": {"torch": torch.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__},
    "baseline_config": BASELINE_CONFIG, "primary_config": PRIMARY_CONFIG, "changed_field": changed_fields[0],
    "baseline_metrics": baseline_summary, "primary_metrics": {"accuracy": primary_validation["accuracy"], "macro_f1": primary_validation["macro_f1"], "worst_class_recall": float(primary_recall.min())},
    "comparison": comparison, "resources": primary_resources, "recovery_used": recovery_used,
}
evidence_board = {
    "baseline_facts": "", "diagnosis": "", "threatening_evidence": "", "prediction": "",
    "observed_result": "", "interpretation": "", "cost_or_constraint": "", "next_step": "",
}
assert all(value.strip() for value in evidence_board.values())
serialized_ledger = json.dumps(experiment_ledger, sort_keys=True, indent=2); serialized_board = json.dumps(evidence_board, sort_keys=True, indent=2)
json.loads(serialized_ledger); json.loads(serialized_board)

## Decision Gate Before Test

Accept, reject, or revise the diagnosis. State whether the case target was met, what regression is unacceptable, and what choice is now frozen. This decision consumes validation evidence; test remains untouched until the gate passes.

In [ ]:
decision_record = {
    "diagnosis_status": "", "target_met_or_not": "", "supporting_delta": "",
    "unacceptable_regression_check": "", "frozen_model_choice": "", "why_no_more_validation_tuning": "",
}
assert all(value.strip() for value in decision_record.values())
TEST_AUTHORIZED = True
X_test, y_test = access_test_split()
print("Test authorized after decision; count:", len(y_test))

In [ ]:
if primary_run is not None:
    test_metrics = evaluate_model(primary_run["model"], X_test, y_test)
    authorized_test_record = {"accuracy": test_metrics["accuracy"], "macro_f1": test_metrics["macro_f1"], "used_for_further_tuning": False}
else:
    authorized_test_record = {"status": "not available from validation-only recovery artifact", "used_for_further_tuning": False}
print("Authorized test record:", authorized_test_record)

## Optional Bounded Follow-Up

Only after the primary decision and test record, name a second intervention. The optional branch may run once for learning, but it must not replace the frozen test claim or trigger another test evaluation.

In [ ]:
RUN_OPTIONAL_FOLLOW_UP = False
optional_follow_up = {"single_change": "", "reason": "", "predicted_validation_evidence": "", "why_test_stays_closed": ""}
if RUN_OPTIONAL_FOLLOW_UP:
    assert all(value.strip() for value in optional_follow_up.values())
    OPTIONAL_CONFIG = dict(PRIMARY_CONFIG)
    # TODO: choose one bounded follow-up field before enabling this branch.
    optional_changed = [field for field in MAJOR_FIELDS if OPTIONAL_CONFIG[field] != PRIMARY_CONFIG[field]]
    assert len(optional_changed) == 1
    optional_run = run_experiment(OPTIONAL_CONFIG, case_train_indices, val_indices)
    print("Optional validation only:", optional_run["validation"]["accuracy"])

## Defense Fields

Prepare a five-minute team defense: what was wrong, what evidence supports and threatens that view, what changed, what happened, why, what it cost, and what should happen next. Do not add an individual transfer response to the team artifact. After the team defense, each participant completes [CHECK-D4-04](../day-4/assessments/day-4-checks.md#check-d4-04---individual-frontier-transfer) separately.

In [ ]:
defense = {
    "problem_and_consequence": "", "primary_diagnosis": "", "supporting_evidence": "",
    "disconfirming_or_threatening_evidence": "", "single_intervention": "", "observed_result": "",
    "mechanism_explanation": "", "efficiency_and_reproducibility": "", "test_statement": "",
    "next_experiment": "",
}
assert all(value.strip() for value in defense.values())

## Challenge and Reflection

Challenge your own case: write the strongest alternative explanation that remains consistent with the evidence and the cheapest new observation that would separate it from your current view. Then identify one way the small digits framing fails to represent a production routing system.

In [ ]:
reflection = {"strongest_alternative": "", "cheapest_discriminating_observation": "", "simulation_limit": "", "what_you_would_monitor": ""}
assert all(value.strip() for value in reflection.values())

## Takeaways and Troubleshooting

- The deliverable is a defensible decision trail, not only a score delta.
- Test is a limited final estimate, not an iterative diagnosis surface.
- One requested evidence card and one primary intervention enforce information discipline.
- Recovery artifacts preserve validation defense after technical failure but deliberately contain no test result.

| Symptom | Likely cause | Recovery |
|---|---|---|
| Test access raises `PermissionError` | Decision gate incomplete | Finish and freeze the decision record |
| Recovery fingerprint mismatch | Cached run does not match committed config | Use the live run or defend baseline evidence; do not relabel recovery |
| More than one changed field | Confounded intervention | Reset from the assigned baseline config |
| Profile metrics differ | Split/seed/config changed | Restore manifest split hash and assigned settings |
| Runtime approaches 12 minutes | Optional or epoch budget expanded | Stop after the primary bounded run |

In [ ]:
assert split_hash == manifest["fixed_split"]["hash"] and TEST_AUTHORIZED
assert len(changed_fields) == 1
json.loads(serialized_ledger); json.loads(serialized_board)
assert all(value.strip() for value in defense.values())
assert all(value.strip() for value in reflection.values())
print(f"LAB-D4-04 checkpoint passed for profile {PROFILE_ID} in {time.perf_counter() - started_capstone:.2f}s: diagnosis gated, primary run recorded, test lock honored, and defense complete.")

## Submit

Return to the [Day 4 capstone debrief](../day-4/student-guide/day-4-student-guide.md#capstone-debrief) and check every field against the [Capstone Rubric](capstone-rubric.md). Keep the serialized ledger and evidence board with your participant submission.